# Clase 201 — Serverless ML: AWS Lambda + GCP Cloud Functions

Notebook **declarativo + cost calculator**. Genera el Dockerfile/handler para Lambda y el script `main.py` para Cloud Functions, y simula el costo mensual de cada arquitectura.

## 1. Lambda Container Image

In [ ]:
lambda_dockerfile = '''\
FROM public.ecr.aws/lambda/python:3.12

COPY requirements.txt ./
RUN pip install --no-cache-dir -r requirements.txt

COPY app.py model.pkl ./

CMD ["app.lambda_handler"]
'''
print('# Dockerfile (Lambda)')
print(lambda_dockerfile)

In [ ]:
lambda_handler = '''\
import json, joblib, numpy as np

# Module-level: corre UNA vez por instancia (cold start), no por request.
MODEL = joblib.load("/var/task/model.pkl")

def lambda_handler(event, context):
    body = event.get("body")
    if isinstance(body, str):
        body = json.loads(body)
    features = body.get("features", [])
    if len(features) != 4 or any(x < 0 for x in features):
        return {"statusCode": 422, "body": json.dumps({"error": "invalid features"})}
    pred = int(MODEL.predict(np.asarray(features).reshape(1, -1))[0])
    return {"statusCode": 200, "body": json.dumps({"class": pred}),
            "headers": {"content-type": "application/json"}}
'''
print('# app.py (Lambda handler)')
print(lambda_handler)

In [ ]:
lambda_deploy = '''\
# Build + push + create
REGION=us-east-1
ACCOUNT=$(aws sts get-caller-identity --query Account --output text)
REPO=iris-lambda

aws ecr create-repository --repository-name $REPO --region $REGION || true
aws ecr get-login-password --region $REGION | docker login --username AWS \\
    --password-stdin $ACCOUNT.dkr.ecr.$REGION.amazonaws.com

docker build -t $REPO:v1 .
docker tag $REPO:v1 $ACCOUNT.dkr.ecr.$REGION.amazonaws.com/$REPO:v1
docker push $ACCOUNT.dkr.ecr.$REGION.amazonaws.com/$REPO:v1

aws lambda create-function \\
    --function-name iris-predict \\
    --package-type Image \\
    --code ImageUri=$ACCOUNT.dkr.ecr.$REGION.amazonaws.com/$REPO:v1 \\
    --role arn:aws:iam::$ACCOUNT:role/lambda-exec \\
    --timeout 30 --memory-size 512

# Mitigar cold start (opcional, cuesta):
aws lambda put-provisioned-concurrency-config \\
    --function-name iris-predict --qualifier 1 \\
    --provisioned-concurrent-executions 2
'''
print(lambda_deploy)

## 2. Cloud Functions 2nd gen

In [ ]:
gcf_main = '''\
# main.py
import functions_framework, joblib, numpy as np

MODEL = joblib.load("model.pkl")   # cargado 1 vez por instancia

@functions_framework.http
def predict(request):
    data = request.get_json(silent=True) or {}
    features = data.get("features", [])
    if len(features) != 4 or any(x < 0 for x in features):
        return ({"error": "invalid features"}, 422)
    pred = int(MODEL.predict(np.asarray(features).reshape(1, -1))[0])
    return {"class": pred}
'''
print('# main.py + requirements.txt')
print(gcf_main)

gcf_deploy = '''\
gcloud functions deploy iris-predict \\
    --gen2 --runtime=python312 --region=us-central1 \\
    --source=. --entry-point=predict \\
    --trigger-http --allow-unauthenticated \\
    --memory=512Mi --timeout=60s \\
    --min-instances=1 --max-instances=20
'''
print(gcf_deploy)

## 3. Cost calculator — Lambda vs K8s

Precios aproximados (junio 2026, us-east-1):
- Lambda on-demand: $0.20 / M requests + $0.0000166667 / GB-s
- Lambda Provisioned Concurrency: $0.0000041667 / GB-s (always-on) + above per-request
- t3.medium on-demand (4 GB, 2 vCPU): $0.0416/h ≈ $30/mes

In [ ]:
def cost_lambda(rps, dur_ms, mem_mb, days=30, pc=0):
    """Lambda monthly cost in USD. pc = provisioned concurrency instances."""
    invocs = rps * 86400 * days
    gb_s = invocs * (dur_ms / 1000) * (mem_mb / 1024)
    request_cost = invocs / 1_000_000 * 0.20
    duration_cost = gb_s * 0.0000166667
    pc_cost = pc * (mem_mb / 1024) * 86400 * days * 0.0000041667 if pc else 0
    return request_cost + duration_cost + pc_cost

def cost_k8s(pods=3, instance_usd_month=30):
    return pods * instance_usd_month

scenarios = [
    ('bursty 1 req/s mean', 1, 80, 512),
    ('moderado 10 req/s', 10, 80, 512),
    ('alto 100 req/s', 100, 80, 512),
    ('muy alto 1000 req/s', 1000, 80, 512),
]
print(f'{"scenario":30} {"Lambda":>10} {"Lambda+PC=2":>14} {"K8s 3pods":>12}')
for name, rps, dur, mem in scenarios:
    l = cost_lambda(rps, dur, mem)
    l_pc = cost_lambda(rps, dur, mem, pc=2)
    k = cost_k8s(3)
    print(f'{name:30} ${l:>9.0f} ${l_pc:>13.0f} ${k:>11.0f}')

print('\n→ Lambda gana a baja carga; K8s gana cuando rps sostenido es alto.')

## Ejercicio guiado

1. Deployá la Lambda real (requiere cuenta AWS). Medí `InitDuration` en CloudWatch — primera invocación cold, siguientes warm.
2. Activá Provisioned Concurrency=2. Repetí test. Compará p99 latency con/sin PC. Compará costo.
3. Deployá la Cloud Function equivalente. Compará cold start con Lambda (suele ser similar en 2nd gen).
4. Modificá `cost_lambda` para sumar API Gateway ($1/M requests). Calculá el punto donde Lambda Function URL (gratis) cambia la economía.
5. Para tu caso real (estimá `rps`, `dur_ms`, `mem_mb`): decidí Lambda vs ECS Fargate vs K8s.

## Conclusiones

- Serverless gana en: tráfico bursty, equipo chico, modelo <1 GB, latencia tolerante.
- Cold start es el villano — mitigá con module-level init + (opcional) PC/min-instances.
- Punto de cruce con K8s: ~100-500 req/s sostenido (depende del modelo y región).
- Container Image hasta 10 GB es el único formato razonable para ML hoy.

## ✅ Soluciones de los ejercicios

Soluciones de los 5 ejercicios del README. El deploy real corre en **AWS Lambda / GCP Cloud Functions** (infra externa), así que Dockerfiles y comandos `aws`/`gcloud` se muestran como referencia, y ejecutamos los *conceptos* cuantificables: la diferencia **cold vs warm start** y — lo más valioso — el **modelo de costo** que decide serverless vs un pod 24/7 (pura aritmética, ejecutable).

In [ ]:
import math
print('conceptos ejecutables: cold/warm start + cross-over de costo.')

### Ejercicio 1 — Lambda con container image (referencia)

Base `public.ecr.aws/lambda/python:3.12`, un `lambda_handler(event, context)` y el `model.pkl`. Se pushea a ECR y se crea la función con `--package-type Image`.

In [ ]:
dockerfile = '''
FROM public.ecr.aws/lambda/python:3.12
COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt
COPY app.py model.pkl ${LAMBDA_TASK_ROOT}/
CMD ["app.lambda_handler"]
'''
handler = '''
import json, joblib, numpy as np
model = joblib.load("model.pkl")          # se carga en el INIT (fuera del handler) -> se reusa en warm
def lambda_handler(event, context):
    body = json.loads(event["body"])
    x = np.array(body["features"]).reshape(1, -1)
    return {"statusCode": 200, "body": json.dumps({"prediction": int(model.predict(x)[0])})}
'''
CLI = """
aws ecr create-repository --repository-name iris
docker build -t iris . && docker tag iris:latest <acct>.dkr.ecr.<region>.amazonaws.com/iris:v1
docker push <acct>.dkr.ecr.<region>.amazonaws.com/iris:v1
aws lambda create-function --function-name iris --package-type Image \\
  --code ImageUri=<acct>.dkr.ecr.<region>.amazonaws.com/iris:v1 --role <role-arn>
"""
assert 'lambda_handler' in handler and '${LAMBDA_TASK_ROOT}' in dockerfile
print('OK — el modelo se carga en el init (module scope) para reusarse entre invocaciones warm.')

### Ejercicio 2 — API Gateway + cold vs warm

Primera invocación paga el **cold start** (arrancar el contenedor + importar libs + cargar modelo); las siguientes reusan el entorno "caliente". Lo modelamos.

In [ ]:
COLD_INIT_S = 2.5     # arrancar runtime + import sklearn + joblib.load
WARM_HANDLER_S = 0.02 # solo la predicción

def invocation_latency(is_cold):
    return (COLD_INIT_S + WARM_HANDLER_S) if is_cold else WARM_HANDLER_S

lat_first  = invocation_latency(is_cold=True)
lat_second = invocation_latency(is_cold=False)
print(f'1ª llamada (cold): {lat_first*1000:.0f} ms')
print(f'2ª llamada (warm): {lat_second*1000:.0f} ms')
assert lat_first > 100 * lat_second, 'el cold start domina la primera invocación'
print('OK — el cold start castiga la primera request tras un periodo de inactividad.')

### Ejercicio 3 — Provisioned Concurrency elimina el cold start

Con `provisioned-concurrency=N`, AWS mantiene N entornos ya inicializados: nunca hay cold start (a cambio de pagar por tenerlos vivos).

In [ ]:
CLI = "aws lambda put-provisioned-concurrency-config --function-name iris --qualifier PROD --provisioned-concurrent-executions 1"
def latency_with_provisioned(is_cold, provisioned):
    return WARM_HANDLER_S if provisioned else invocation_latency(is_cold)

assert latency_with_provisioned(is_cold=True, provisioned=True) == WARM_HANDLER_S
print(f'con provisioned concurrency: {latency_with_provisioned(True, True)*1000:.0f} ms (sin cold start)')
print('OK — trade-off: pagás por mantener entornos calientes vs latencia predecible.')

### Ejercicio 4 — Cloud Functions equivalente (referencia)

GCP Cloud Functions 2nd gen. `--min-instances=1` es el equivalente a provisioned concurrency (mantiene 1 instancia caliente).

In [ ]:
CLI = """
gcloud functions deploy iris --gen2 --runtime=python312 --trigger-http \\
  --source=. --entry-point=predict --memory=512Mi --min-instances=1 --region=us-central1
"""
print(CLI)
print('--min-instances=1  ==  provisioned concurrency de Lambda (misma idea, otro proveedor).')

### Ejercicio 5 — Costo: cross-over serverless vs pod 24/7 (ejecutable)

El modelo de negocio de serverless: pagás por request; un pod 24/7 paga por hora esté o no usado. A tráfico bajo gana serverless; a tráfico alto y sostenido gana el pod. Calculamos el punto de cruce.

In [ ]:
# Precios aproximados (orden de magnitud, us-east-1)
LAMBDA_PER_REQUEST = 0.20 / 1_000_000      # $ por request
LAMBDA_PER_GB_S    = 0.0000166667           # $ por GB-segundo
MEM_GB = 0.512
DURATION_S = 0.08                           # p99 80 ms
POD_VCPU_HOUR = 0.04                         # 1 vCPU-hora (~4 vCPU pod = 0.16/h)
POD_VCPUS = 4
SECONDS_MONTH = 30 * 24 * 3600

def lambda_monthly(rps):
    reqs = rps * SECONDS_MONTH
    gb_s = reqs * DURATION_S * MEM_GB
    return reqs * LAMBDA_PER_REQUEST + gb_s * LAMBDA_PER_GB_S

def pod_monthly():
    return POD_VCPUS * POD_VCPU_HOUR * 24 * 30   # costo fijo, 24/7

pod_cost = pod_monthly()
print(f'pod 4 vCPU 24/7: ${pod_cost:,.0f}/mes (fijo)')
for rps in [1, 10, 50, 100, 300]:
    lc = lambda_monthly(rps)
    winner = 'lambda' if lc < pod_cost else 'pod'
    print(f'  {rps:4} rps -> lambda ${lc:8,.0f}/mes  => gana {winner}')

# cross-over: rps donde lambda == pod
per_rps = lambda_monthly(1)
crossover = pod_cost / per_rps
print(f'\ncross-over ~ {crossover:.0f} rps sostenidos: por debajo conviene serverless, por encima el pod.')
assert lambda_monthly(1) < pod_cost < lambda_monthly(1000)
print('OK — regla: tráfico bajo/espádico => serverless; alto y constante => pod dedicado.')